# Phase 2 — LSTM Model for RUL Prediction
**NASA C-MAPSS FD001 Dataset**

This notebook trains an LSTM neural network to predict Remaining Useful Life (RUL) of aircraft turbofan engines.

Run cells top to bottom in order.

## Cell 1 — Imports & Device Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
import itertools
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Imports successful!')
print(f'🖥️  Using device: {device}')
print(f'🔥 PyTorch version: {torch.__version__}')
if torch.cuda.is_available():
    print(f'🎮 GPU: {torch.cuda.get_device_name(0)}')

## Cell 2 — Load & Prepare Data

In [ ]:
columns = ['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + \
          [f'sensor_{i}' for i in range(1, 22)]

train_df = pd.read_csv('../data/raw/train_FD001.txt', sep=r'\s+', header=None, names=columns)
test_df  = pd.read_csv('../data/raw/test_FD001.txt',  sep=r'\s+', header=None, names=columns)

# Drop dead + weak sensors
drop_sensors = ['sensor_1','sensor_5','sensor_6','sensor_10',
                'sensor_16','sensor_18','sensor_19','sensor_8','sensor_13','sensor_15']
train_df.drop(columns=drop_sensors, inplace=True)
test_df.drop(columns=drop_sensors, inplace=True)

# Add RUL labels
max_cycles = train_df.groupby('engine_id')['cycle'].max().reset_index()
max_cycles.columns = ['engine_id', 'max_cycle']
train_df = train_df.merge(max_cycles, on='engine_id', how='left')
train_df['RUL'] = (train_df['max_cycle'] - train_df['cycle']).clip(upper=125)
train_df.drop(columns=['max_cycle'], inplace=True)

feature_cols = ['cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3',
                'sensor_2','sensor_3','sensor_4','sensor_7','sensor_9',
                'sensor_11','sensor_12','sensor_14','sensor_17','sensor_20','sensor_21']

print(f'✅ Data loaded! Shape: {train_df.shape}')
print(f'Features: {len(feature_cols)}')
train_df[['engine_id','cycle','RUL']].head()

## Cell 3 — Normalize + Create Sliding Windows

In [ ]:
WINDOW_SIZE = 30

# Fit and apply MinMaxScaler
scaler = MinMaxScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])

# Save scaler immediately after fitting
joblib.dump(scaler, '../models/minmax_scaler.pkl')
print('✅ Scaler saved to models/minmax_scaler.pkl')

# Save feature columns
joblib.dump(feature_cols, '../models/feature_cols.pkl')
print('✅ Feature cols saved to models/feature_cols.pkl')

# Create sliding windows
def create_windows(df, window_size, feature_cols):
    X, y, groups = [], [], []
    for eng_id in df['engine_id'].unique():
        eng_data = df[df['engine_id'] == eng_id].reset_index(drop=True)
        for i in range(len(eng_data) - window_size + 1):
            X.append(eng_data[feature_cols].iloc[i:i+window_size].values)
            y.append(eng_data['RUL'].iloc[i+window_size-1])
            groups.append(eng_id)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), np.array(groups)

X, y, engine_ids = create_windows(train_df, WINDOW_SIZE, feature_cols)

# Split by engine group (no data leakage)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(X, groups=engine_ids))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f'\n✅ Windows created!')
print(f'X_train: {X_train.shape} → (samples, window, features)')
print(f'X_val  : {X_val.shape}')

## Cell 4 — PyTorch Dataset & DataLoaders

In [ ]:
class EngineDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = EngineDataset(X_train, y_train)
val_dataset   = EngineDataset(X_val,   y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0)

print(f'✅ DataLoaders ready!')
print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')

## Cell 5 — LSTM Model Architecture (LeakyReLU)

In [ ]:
class LSTMModelV2(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(LSTMModelV2, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.LeakyReLU(0.01),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        return self.fc(last_out).squeeze()

model = LSTMModelV2(input_size=len(feature_cols)).to(device)
print(f'✅ LSTM Model created!')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(model)

## Cell 6 — Train LSTM

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

EPOCHS = 60
best_val_loss = float('inf')
train_losses, val_losses = [], []

print('🚀 Training LSTM on GPU...')
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            val_loss += criterion(pred, y_batch).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../models/lstm_best.pt')

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_loss:.2f} | Val: {val_loss:.2f}')

print(f'\n✅ Training complete! Best Val Loss: {best_val_loss:.2f}')

## Cell 7 — Evaluate LSTM v1

In [ ]:
model.load_state_dict(torch.load('../models/lstm_best.pt', map_location=device))
model.eval()

y_pred_list = []
with torch.no_grad():
    for X_batch, _ in val_loader:
        X_batch = X_batch.to(device)
        pred = model(X_batch)
        y_pred_list.extend(pred.cpu().numpy())

y_pred = np.clip(np.array(y_pred_list), 0, 125)

rmse_v1 = np.sqrt(mean_squared_error(y_val, y_pred))
mae_v1  = mean_absolute_error(y_val, y_pred)

print(f'📊 LSTM v1 Results:')
print(f'   RMSE : {rmse_v1:.2f} cycles')
print(f'   MAE  : {mae_v1:.2f} cycles')
print(f'\n📊 vs RF baseline:')
print(f'   RF   RMSE : 15.37 cycles')
print(f'   LSTM RMSE : {rmse_v1:.2f} cycles')
if rmse_v1 < 15.37:
    print(f'   ✅ LSTM beats RF by {15.37-rmse_v1:.2f} cycles!')
else:
    print(f'   ❌ RF still better — need tuning')

## Cell 8 — Hyperparameter Tuning

In [ ]:
param_grid = {
    'hidden_size' : [64, 128, 256],
    'num_layers'  : [2, 3],
    'dropout'     : [0.2, 0.3],
    'lr'          : [0.001, 0.0005],
}

results = []
best_overall = float('inf')
best_params  = None

combos = list(itertools.product(
    param_grid['hidden_size'],
    param_grid['num_layers'],
    param_grid['dropout'],
    param_grid['lr']
))

print(f'🔍 Total combinations: {len(combos)}')
print('⏳ Running search...\n')

for i, (hidden, layers, drop, lr) in enumerate(combos):
    m = LSTMModelV2(
        input_size=len(feature_cols),
        hidden_size=hidden,
        num_layers=layers,
        dropout=drop
    ).to(device)

    opt  = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.MSELoss()

    for epoch in range(30):
        m.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            opt.zero_grad()
            pred = m(X_batch)
            loss = crit(pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()

    m.eval()
    preds = []
    with torch.no_grad():
        for X_batch, _ in val_loader:
            preds.extend(m(X_batch.to(device)).cpu().numpy())
    rmse = np.sqrt(mean_squared_error(y_val, np.clip(preds, 0, 125)))

    results.append({'hidden': hidden, 'layers': layers, 'dropout': drop, 'lr': lr, 'rmse': rmse})

    if rmse < best_overall:
        best_overall = rmse
        best_params  = {'hidden': hidden, 'layers': layers, 'dropout': drop, 'lr': lr}
        torch.save(m.state_dict(), '../models/lstm_tuned.pt')

    print(f'[{i+1:2d}/{len(combos)}] hidden={hidden} layers={layers} drop={drop} lr={lr} → RMSE: {rmse:.2f}')

print(f'\n✅ Best params: {best_params}')
print(f'🏆 Best RMSE  : {best_overall:.2f} cycles')

## Cell 9 — Final Results & Save Everything

In [ ]:
results_df = pd.DataFrame(results).sort_values('rmse')
print('📊 Top 5 Configurations:')
print(results_df.head())

print(f'\n📊 Full Comparison:')
print(f'{"Model":<25} {"RMSE":>8}')
print('-' * 35)
print(f'{"RF (tuned)":<25} {"15.37":>8}')
print(f'{"LSTM v1":<25} {rmse_v1:>8.2f}')
print(f'{"LSTM Tuned":<25} {best_overall:>8.2f}')

improvement = ((15.37 - best_overall) / 15.37) * 100
print(f'\n🚀 Improvement over RF: {improvement:.1f}%')

## Cell 10 — Save All Model Files

In [ ]:
# Verify all files are saved
import os

files_to_check = [
    '../models/minmax_scaler.pkl',
    '../models/feature_cols.pkl',
    '../models/lstm_best.pt',
    '../models/lstm_tuned.pt',
]

print('📁 Model files status:')
for f in files_to_check:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) / 1024 if exists else 0
    status = '✅' if exists else '❌ MISSING'
    print(f'   {status} {os.path.basename(f):30s} {size:.1f} KB')

print('\n✅ All files ready for deployment!')
print('Next: git add -f models/ && git push')

## Cell 11 — Training Curves Visualization

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', color='steelblue')
plt.plot(val_losses,   label='Val Loss',   color='darkorange')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('LSTM Training Curves')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(y_val, y_pred, alpha=0.3, color='steelblue', s=10)
plt.plot([0, 125], [0, 125], 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual RUL')
plt.ylabel('Predicted RUL')
plt.title(f'Actual vs Predicted RUL (RMSE={rmse_v1:.2f})')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()